In [1]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, shutil, hashlib, subprocess
from pathlib import Path
DRIVE_ROOT=Path('/content/drive/MyDrive'); PARENT_DIR=DRIVE_ROOT/'CALSHIFT_Research'
PROJECT_ROOT=PARENT_DIR/'calshift-research'; CRED_DIR=DRIVE_ROOT/'.gitcreds'
subprocess.run(['git','config','--global','user.name','Md Anas Biswas'],check=False)
subprocess.run(['git','config','--global','user.email','anasbiswas@gmail.com'],check=False)
subprocess.run(['git','config','--global','credential.helper','store'],check=False)
for fn,dest in [('.git-credentials','/root/.git-credentials'),('.gitconfig','/root/.gitconfig')]:
    for cand in (PARENT_DIR/fn, CRED_DIR/fn):
        if cand.exists(): shutil.copy(cand,dest); os.chmod(dest,0o600); break
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT/'src'))
subprocess.run(['git','pull','--ff-only','--quiet'],check=False)
import importlib
if 'config' in sys.modules: importlib.reload(sys.modules['config'])
import config
import numpy as np, pandas as pd
print('ready:', os.getcwd())


Mounted at /content/drive
ready: /content/drive/MyDrive/CALSHIFT_Research/calshift-research


In [2]:
# =============================================================================
# Cell 2 - mechanism function (deterministic mid-U APS score, for a reproducible
# score DISTRIBUTION; nb17 remains the randomized coverage of record). Given one
# model's calibrated probs on source pool and target, per class it fixes the SHC
# quantile from source true-class scores and measures target coverage + how far
# the target true-class score distribution moved.
# =============================================================================
ALPHA = config.ALPHA_PRIMARY
def aps_true_scores(P, y_int, cls):
    mk=(y_int==cls); Pm=P[mk]
    if len(Pm)==0: return np.array([])
    order=np.argsort(-Pm,axis=1); sp=np.take_along_axis(Pm,order,1); cum=np.cumsum(sp,1)
    ss=cum-0.5*sp; pos=np.argmax(order==cls,axis=1)
    return ss[np.arange(len(Pm)),pos]
def qhat(s,a):
    n=len(s); return np.inf if n<1 else float(np.quantile(s,min(np.ceil((n+1)*(1-a))/n,1.0),method='higher'))
def class_shift_row(dataset, arch, cls_name, cls_idx, P_src, y_src, P_tgt, y_tgt):
    ssrc=aps_true_scores(P_src,y_src,cls_idx); stgt=aps_true_scores(P_tgt,y_tgt,cls_idx)
    if len(ssrc)<5 or len(stgt)<5: return None
    q=qhat(ssrc,ALPHA); shc=float(np.mean(stgt<=q))
    med=float(np.median(stgt)-np.median(ssrc)); q90=float(np.quantile(stgt,0.9)-np.quantile(ssrc,0.9))
    allv=np.sort(np.concatenate([ssrc,stgt]))
    cs=np.searchsorted(np.sort(ssrc),allv,side='right')/len(ssrc)
    ct=np.searchsorted(np.sort(stgt),allv,side='right')/len(stgt)
    ks=float(np.max(np.abs(cs-ct)))
    return {'dataset':dataset,'arch':arch,'class':cls_name,'n_src':len(ssrc),'n_tgt':len(stgt),
            'q_src':round(q,4),'SHC_coverage':round(shc,4),'undercoverage':round((1-ALPHA)-shc,4),
            'median_score_shift':round(med,4),'q90_score_shift':round(q90,4),'score_KS':round(ks,4)}

def load_probs(f):
    d=np.load(f); return d['srcpool'], d['target']
print('mechanism function ready (per-architecture); alpha =', ALPHA)


mechanism function ready (per-architecture); alpha = 0.05


In [3]:
# =============================================================================
# Cell 3 - PER-ARCHITECTURE score-shift rows (no cross-architecture averaging).
# For each architecture we average only across the 10 seeds of THAT architecture,
# so the score distribution stays a real one. CIC per realization, UGR single pair.
# =============================================================================
ARCHS=['rf','xgb','mlp']
def avg_arch(prob_dir, key, arch):
    files=sorted(Path(prob_dir).glob(f'{key}{arch}__seed*.npz'))
    if not files: return None,None,0
    sp=tg=None; n0=None; k=0
    for f in files:
        s,t=load_probs(f)
        if n0 is None: n0=(s.shape,t.shape)
        elif (s.shape,t.shape)!=n0:  # guard: mismatched shapes across seeds
            raise ValueError(f'shape mismatch in {f.name}: {(s.shape,t.shape)} vs {n0}')
        sp=s if sp is None else sp+s; tg=t if tg is None else tg+t; k+=1
    return sp/k, tg/k, k

rows=[]
# ---- CIC ----
cic=pd.read_parquet(config.INTERIM_DIR/'cicids2017_primary.parquet')
wed=cic[cic['day']=='wednesday'].reset_index(drop=True); wed=wed[wed['label'].isin(['DoS','Benign'])].reset_index(drop=True)
CICC=['Benign','DoS']; CIC_PROBS=config.DATA_DIR/'cic_probs'
REAL=['R1_holdout_Slowhttptest','R2_holdout_Slowloris','R3_holdout_GoldenEye',
      'R4_holdout_Slowloris_Slowhttptest','R5_holdout_GoldenEye_Slowloris']
def lab_cic(idx): return (wed.loc[idx,'label'].to_numpy()=='DoS').astype(int)
for name in REAL:
    spx=np.load(config.PROC_DIR/f'cic_{name}_srcpool_idx.npy'); tgx=np.load(config.PROC_DIR/f'cic_{name}_target_idx.npy')
    ysp,ytg=lab_cic(spx),lab_cic(tgx)
    for arch in ARCHS:
        Psp,Ptg,k=avg_arch(CIC_PROBS,f'{name}__',arch)
        if k==0: print('WARN no CIC files for',name,arch); continue
        for ci,cn in enumerate(CICC):
            r=class_shift_row(f'cicids2017:{name}',arch,cn,ci,Psp,ysp,Ptg,ytg)
            if r: rows.append(r)
# ---- UGR ----
UGR=config.DATASETS_DIR/'ugr16'; usrc=pd.read_parquet(UGR/'july_week5.parquet'); utgt=pd.read_parquet(UGR/'august_week1.parquet')
for dd in (usrc,utgt): dd['label']=dd['label'].astype(str).str.strip().str.lower()
UK=['background','dos','scan11','scan44','nerisbotnet']
usrc=usrc[usrc.label.isin(UK)].reset_index(drop=True); utgt=utgt[utgt.label.isin(UK)].reset_index(drop=True)
UCL=sorted(UK); U2I={c:i for i,c in enumerate(UCL)}
def strat(df,fr,seed,col='label'):
    rng=np.random.default_rng(seed); nm=list(fr); f=np.array([fr[k] for k in nm],float); big=nm[int(np.argmax(f))]
    a=pd.Series(index=df.index,dtype=object)
    for _,s in df.groupby(col,sort=True):
        idx=s.index.to_numpy().copy(); rng.shuffle(idx); n=len(idx)
        c=np.floor(f*n).astype(int); c[nm.index(big)]+=n-c.sum(); kk=0
        for a2,q in zip(nm,c): a.loc[idx[kk:kk+q]]=a2; kk+=q
    return a
usrc=usrc.assign(partition=strat(usrc,config.SPLIT_FRACTIONS,20260725).values)
y_sp=usrc[usrc.partition=='source_cal_pool']['label'].map(U2I).to_numpy(); y_tg=utgt['label'].map(U2I).to_numpy()
for arch in ARCHS:
    Psp,Ptg,k=avg_arch(config.DATA_DIR/'ugr16_probs','ugr16__',arch)
    if k==0: print('WARN no UGR files for',arch); continue
    for ci,cn in enumerate(UCL):
        r=class_shift_row('ugr16:july_to_august',arch,cn,ci,Psp,y_sp,Ptg,y_tg)
        if r: rows.append(r)
shift_tbl=pd.DataFrame(rows)
print('rows:', len(shift_tbl), '| archs:', sorted(shift_tbl.arch.unique()))
print('\nUGR, per architecture (sharp case):')
print(shift_tbl[shift_tbl.dataset.str.startswith('ugr16')][
      ['arch','class','SHC_coverage','undercoverage','q90_score_shift','score_KS']].to_string(index=False))


rows: 45 | archs: ['mlp', 'rf', 'xgb']

UGR, per architecture (sharp case):
arch       class  SHC_coverage  undercoverage  q90_score_shift  score_KS
  rf  background        0.9973        -0.0473           0.0000    0.0586
  rf         dos        1.0000        -0.0500           0.0000    0.0062
  rf nerisbotnet        0.9891        -0.0391           0.0000    0.0892
  rf      scan11        0.3916         0.5584           0.4998    0.6056
  rf      scan44        0.7368         0.2132           0.4999    0.2604
 xgb  background        0.9971        -0.0471           0.0000    0.0087
 xgb         dos        1.0000        -0.0500           0.0000    0.0042
 xgb nerisbotnet        0.9899        -0.0399           0.0000    0.0769
 xgb      scan11        0.4301         0.5199           0.4999    0.5587
 xgb      scan44        0.8211         0.1289           0.2435    0.1738
 mlp  background        0.9651        -0.0151           0.0000    0.0924
 mlp         dos        1.0000        -0.0500   

In [4]:
# =============================================================================
# Cell 4 - THE TEST, held per architecture (no blending): does score movement
# predict undercoverage across classes, within each architecture and pooled?
# =============================================================================
from scipy import stats
t=shift_tbl.copy()
print('Spearman( score movement , undercoverage ), per architecture:')
res_by_arch={}
for arch in sorted(t.arch.unique()):
    ta=t[t.arch==arch]
    out={}
    for mv in ['score_KS','q90_score_shift','median_score_shift']:
        rho,p=stats.spearmanr(ta[mv],ta['undercoverage']); out[mv]={'rho':round(float(rho),3),'p':float(p)}
    res_by_arch[arch]=out
    print(f'  [{arch}] KS: rho={out["score_KS"]["rho"]:+.3f} p={out["score_KS"]["p"]:.3g}'
          f'   q90: rho={out["q90_score_shift"]["rho"]:+.3f}'
          f'   median: rho={out["median_score_shift"]["rho"]:+.3f}  (n={len(ta)})')

rho_all,p_all=stats.spearmanr(t['score_KS'],t['undercoverage'])
print(f'\npooled across all rows: Spearman(score_KS, undercoverage) rho={rho_all:+.3f} p={p_all:.3g} (n={len(t)})')
print('\nUGR mean over architectures (class-level mechanism):')
um=t[t.dataset.str.startswith('ugr16')].groupby('class').agg(
     SHC_coverage=('SHC_coverage','mean'),undercoverage=('undercoverage','mean'),
     q90_score_shift=('q90_score_shift','mean'),score_KS=('score_KS','mean')).round(4).sort_values('undercoverage',ascending=False)
print(um.to_string())

verdict={'test':'does source->target true-class score movement predict SHC undercoverage across classes',
         'per_architecture_spearman_KS_vs_undercoverage':{a:res_by_arch[a]['score_KS'] for a in res_by_arch},
         'pooled_spearman_KS':{'rho':round(float(rho_all),4),'p':float(p_all),'n':int(len(t))},
         'reads':('positive rho in every architecture => score movement drives undercoverage, robustly, '
                  'not an artifact of one model or of cross-architecture blending; evidence base for a '
                  'label-free score-movement detector.'),
         'ugr_sharp_case':'nerisbotnet stable+holds vs scan11/scan44 moved+collapse'}


Spearman( score movement , undercoverage ), per architecture:
  [mlp] KS: rho=+0.865 p=3.15e-05   q90: rho=+0.923   median: rho=+0.285  (n=15)
  [rf] KS: rho=+0.931 p=4.59e-07   q90: rho=+0.875   median: rho=+0.664  (n=15)
  [xgb] KS: rho=+0.876 p=1.86e-05   q90: rho=+0.776   median: rho=+0.184  (n=15)

pooled across all rows: Spearman(score_KS, undercoverage) rho=+0.895 p=1.17e-16 (n=45)

UGR mean over architectures (class-level mechanism):
             SHC_coverage  undercoverage  q90_score_shift  score_KS
class                                                              
scan11             0.4597         0.4903           0.3659    0.5206
scan44             0.7883         0.1617           0.2748    0.1993
nerisbotnet        0.9729        -0.0229           0.0001    0.0922
background         0.9865        -0.0365           0.0000    0.0532
dos                1.0000        -0.0500           0.0000    0.0078


In [5]:
# =============================================================================
# Cell 5 - save table + verdict + figure (points colored by architecture), commit.
# NSL absent: notebooks 01-08 did not save calibrated probs, so its score
# distributions cannot be reconstructed; CIC and UGR carry the mechanism test.
# =============================================================================
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
shift_tbl.to_csv(config.REPORTS_DIR/'score_shift_explainability.csv', index=False)
(config.REPORTS_DIR/'score_shift_verdict.json').write_text(json.dumps(verdict,indent=2))

fig,ax=plt.subplots(figsize=(6.2,4.6))
col={'rf':'tab:blue','xgb':'tab:green','mlp':'tab:red'}
for arch in sorted(shift_tbl.arch.unique()):
    sub=shift_tbl[shift_tbl.arch==arch]
    ax.scatter(sub['q90_score_shift'], sub['undercoverage'], s=42, alpha=0.75, c=col.get(arch,'grey'), label=arch)
for _,r in shift_tbl[(shift_tbl.dataset.str.startswith('ugr16'))&(shift_tbl.arch=='xgb')].iterrows():
    ax.annotate(r['class'], (r['q90_score_shift'], r['undercoverage']), fontsize=7, xytext=(3,3), textcoords='offset points')
ax.axhline(0,color='grey',lw=0.7); ax.set_xlabel('upper-tail true-class score shift (q90, source->target)')
ax.set_ylabel('SHC undercoverage (nominal - coverage)'); ax.legend(title='arch'); ax.set_title('Score movement predicts coverage failure')
fig.tight_layout(); fig.savefig(config.REPORTS_DIR/'score_shift_vs_undercoverage.png', dpi=140)
print('figure saved: score_shift_vs_undercoverage.png')

def git(*a, show=True):
    r=subprocess.run(['git',*a],capture_output=True,text=True)
    if show and (r.stdout or r.stderr): print((r.stdout+r.stderr).strip())
    return r
for s,dd in [('/root/.git-credentials',PARENT_DIR/'.git-credentials'),('/root/.gitconfig',PARENT_DIR/'.gitconfig')]:
    if os.path.exists(s): shutil.copy(s,dd)
os.chdir(PROJECT_ROOT); git('add','-A',show=False)
if git('status','--porcelain',show=False).stdout.strip():
    git('commit','-m','nb20: per-architecture score-distribution explainability (score movement predicts SHC undercoverage)')
    r=git('push','-u','origin','main')
    if r.returncode: print('PUSH FAILED. Commit is safe locally.')
else: print('nothing to commit')
print(git('log','--oneline','-3',show=False).stdout)


figure saved: score_shift_vs_undercoverage.png
[main 62ed688] nb20: per-architecture score-distribution explainability (score movement predicts SHC undercoverage)
 5 files changed, 72 insertions(+), 1 deletion(-)
 create mode 100644 notebooks/20_score_shift_explainability.ipynb
 create mode 100644 reports/score_shift_explainability.csv
 create mode 100644 reports/score_shift_verdict.json
 create mode 100644 reports/score_shift_vs_undercoverage.png
Branch 'main' set up to track remote branch 'main' from 'origin'.
To https://github.com/anasbiswas1/calshift-research.git
   51caaf4..62ed688  main -> main
62ed688 nb20: per-architecture score-distribution explainability (score movement predicts SHC undercoverage)
51caaf4 pooled model: record beta5 (SHC x S_cov) as sign-unstable / not identified across estimators
328fa9d nb19: pooled model (cluster-robust binomial GLM + mixed-model check); beta7 identified, beta5 suggestive

